In [1]:
import pandas as pd
import numpy as np
import plotly.express as px



In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler ,OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor


from sklearn.model_selection import cross_validate,RandomizedSearchCV,GridSearchCV
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor,ColumnTransformer
from category_encoders import BinaryEncoder
import joblib

# loading , understanding , cleaning 

In [3]:
df = pd.read_csv('global_power_plants.csv',index_col=False)
df.head()

,country code,country_long,name of powerplant,capacity in MW,latitude,longitude,primary_fuel,secondary fuel,other_fuel2,other_fuel3,start date,owner of plant,generation_gwh_2021,geolocation_source,estimated_generation_gwh_2021
0,AFG,Afghanistan,Kajaki Hydroelectric Power Plant Afghanistan,33.0,32.322,65.1190,Hydro,NaN,NaN,NaN,NaN,NaN,NaN,GEODB,123.77
1,AFG,Afghanistan,Kandahar DOG,10.0,31.670,65.7950,Solar,NaN,NaN,NaN,NaN,NaN,NaN,Wiki-Solar,18.43
2,AFG,Afghanistan,Kandahar JOL,10.0,31.623,65.7920,Solar,NaN,NaN,NaN,NaN,NaN,NaN,Wiki-Solar,18.64
3,AFG,Afghanistan,Mahipar Hydroelectric Power Plant Afghanistan,66.0,34.556,69.4787,Hydro,NaN,NaN,NaN,NaN,NaN,NaN,GEODB,225.06
4,AFG,Afghanistan,Naghlu Dam Hydroelectric Power Plant Afghanistan,100.0,34.641,69.7170,Hydro,NaN,NaN,NaN,NaN,NaN,NaN,GEODB,406.16


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34936 entries, 0 to 34935
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   country code                   34936 non-null  str    
 1   country_long                   34936 non-null  str    
 2   name of powerplant             34936 non-null  str    
 3   capacity in MW                 34936 non-null  float64
 4   latitude                       34936 non-null  float64
 5   longitude                      34936 non-null  float64
 6   primary_fuel                   34936 non-null  str    
 7   secondary fuel                 1944 non-null   str    
 8   other_fuel2                    276 non-null    str    
 9   other_fuel3                    92 non-null     str    
 10  start date                     17447 non-null  float64
 11  owner of plant                 20868 non-null  str    
 12  generation_gwh_2021            9659 non-null   float64
 1

In [5]:
# changing features names for better readability

df = df.rename(columns ={'country_long':'country','estimated_generation_gwh_2021':'estimated_generation'})



In [6]:
# dropping nulls from the target variable

df = df.dropna(subset=['estimated_generation'],axis = 0).reset_index(drop=True)

In [7]:
# dropping redundant features and features with > 50% missing values

df = df.drop(columns=['country code','secondary fuel','start date',
       'other_fuel2', 'other_fuel3','generation_gwh_2021','name of powerplant',
       'latitude','longitude','owner of plant'])

In [8]:
df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

In [9]:
# filling missing for the deployed dataframe version only
df_deploy = df.copy()
df_deploy['geolocation_source'].fillna(df_deploy['geolocation_source'].mode()[0], inplace=True)

C:\Users\ELkayan\AppData\Local\Temp\ipykernel_5356\3751499038.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_deploy['geolocation_source'].fillna(df_deploy['geolocation_source'].mode()[0], inplace=True)


0               GEODB
1          Wiki-Solar
2          Wiki-Solar
3               GEODB
4               GEODB
             ...     
14258           GEODB
14259           GEODB
14260           GEODB
14261    Power Africa
14262           GEODB
Name: geolocation_source, Length: 14263, dtype: str

In [10]:
df_deploy.info()

<class 'pandas.DataFrame'>
RangeIndex: 14263 entries, 0 to 14262
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   country               14263 non-null  str    
 1   capacity in MW        14263 non-null  float64
 2   primary_fuel          14263 non-null  str    
 3   geolocation_source    14005 non-null  str    
 4   estimated_generation  14263 non-null  float64
dtypes: float64(2), str(3)
memory usage: 1.0 MB


In [68]:
# exporting the clead df 

df_deploy.to_csv('global_power_clean.csv',index=False)

In [69]:
df_deploy.describe(include='number')

,capacity in MW,estimated_generation
count,14263.000000,14263.000000
mean,83.405388,265.851763
std,337.113067,1262.610693
min,1.000000,1.120000
25%,5.000000,9.770000
50%,13.600000,32.290000
75%,49.800000,125.720000
max,13050.000000,48675.060000


# EDA

univariate analysis

In [ ]:
for col in df_deploy[['capacity in MW','primary_fuel','estimated_generation']]:
    hist = px.histogram(df_deploy,x=col,marginal = 'rug',nbins=5,
                        title=f"Distrinution of {col}",color_discrete_sequence=['dodgerblue'])
    hist.show()

In [14]:
for col in df_deploy[['capacity in MW','estimated_generation']]:
    box = px.box(df_deploy, y = col,color_discrete_sequence=['dodgerblue'])
    box.show()

bivariate analysis 


In [15]:
# primary_fuel vs. capacity

bar_1= px.bar(df_deploy,x = 'primary_fuel',y='capacity in MW',
            color_discrete_sequence=['dodgerblue'],barmode='group')
bar_1.show()

In [16]:
# primary_fuel vs. estimated generation

bar_2= px.bar(df_deploy,x = 'primary_fuel',y='estimated_generation',
            color_discrete_sequence=['dodgerblue'],barmode='group')
bar_2.show()

In [17]:
# capacity of power plant vs. estimated generation

scatter = px.scatter(df_deploy,y='capacity in MW',x='estimated_generation',trendline='ols',
                     color_discrete_sequence=['steelblue'])
scatter.show()

In [18]:
# top 10 countries by average plant capacity 

group_2 = df_deploy.groupby('country')['capacity in MW'].mean().reset_index().sort_values(by='capacity in MW',ascending=False).head(10)

bar_4 = px.bar(group_2,y = 'capacity in MW',x ='country',title="TOP 10 Countries by  Average Plant Capacity",
               color_discrete_sequence=['skyblue'])

bar_4.show()

In [19]:
# top 10 countries by average Estimated Generation

group_2 = df_deploy.groupby('country')['estimated_generation'].mean().reset_index().sort_values(by='estimated_generation',ascending=False).head(10)

bar_4 = px.bar(group_2,y = 'estimated_generation',x ='country',title="TOP 10 Countries by  Average Estimated Generation",
               color_discrete_sequence=['cornflowerblue'])

bar_4.show()

# Machine Learning 

data preprocessing

In [20]:
y = df['estimated_generation']
x = df.drop(columns = 'estimated_generation')

In [21]:
numerical_features = ['capacity in MW']
one_hot = ['primary_fuel']
binary = ['country','geolocation_source']

In [22]:
numerical_pipeline = Pipeline([('scaling',RobustScaler())])
one_hot_pipeline = Pipeline([('onehot',OneHotEncoder(drop='first'))])

binary_pipeline = Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                            ('binary',BinaryEncoder())])

In [23]:
preprocessing = ColumnTransformer(transformers=[('numerical',numerical_pipeline,numerical_features),
                                                ('one_hot',one_hot_pipeline,one_hot),
                                                ('binary',binary_pipeline,binary)],
                                                remainder='passthrough')
preprocessing

c:\python_hub\power_plants\power\Lib\site-packages\sklearn\externals\_numpydoc\docscrape.py:420: UserWarning: Unknown section Example
  self[section] = content


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('one_hot', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`

Baseline Modelling and cross validation 

In [24]:
model_base = [('KNN',KNeighborsRegressor()),
              ('SVM',SVR()),
              ('Desicion Tree',DecisionTreeRegressor()),
              ('Linear Model',LinearRegression()),
              ('Random Forest',RandomForestRegressor())]

for name,model in model_base:
    
    model_pipeline=Pipeline(steps=[('preprocessing',preprocessing),('model',model)])

    model_pipeline_scaled=TransformedTargetRegressor(regressor=model_pipeline,func=np.log1p,inverse_func=np.expm1)

    result=cross_validate(model_pipeline_scaled,x,y,cv=5,scoring='r2',return_train_score=True,n_jobs=-1)

    print(name)
    print('train_score',result['train_score'].mean())
    print('test_score',result['test_score'].mean())
    print('*'*50)

KNN
train_score 0.9354643499608333
test_score 0.8952728571079824
**************************************************
SVM
train_score 0.669641081992753
test_score 0.5504807578029386
**************************************************
Desicion Tree
train_score 0.998786785828767
test_score 0.8432628637812247
**************************************************
Linear Model
train_score -7.563901828459306e+17
test_score -3.835239420961908e+18
**************************************************
Random Forest
train_score 0.9828910252377092
test_score 0.8883108523896664
**************************************************


KNN is the best performing model and has the lowest gap between test and train scores so it will be selected for final deployment

In [25]:
param_grid = {'model__n_neighbors':[5,7,10,15,20]}

knn_pipeline = Pipeline(steps=[('preprocessing',preprocessing),
                               ('model',KNeighborsRegressor())])

In [26]:
result_1=RandomizedSearchCV(knn_pipeline,param_grid,cv=5,return_train_score=True,scoring='r2',n_jobs=-1)

result_1.fit(x,y)

c:\python_hub\power_plants\power\Lib\site-packages\sklearn\model_selection\_search.py:326: UserWarning: The total space of parameters 5 is smaller than n_iter=10. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...Regressor())])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__n_neighbors': [5, 7, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is ma

In [27]:
print(result_1.best_score_)
print(result_1.best_params_)

0.9005232086831996
{'model__n_neighbors': 7}


In [28]:
result_2=GridSearchCV(knn_pipeline,param_grid,cv=5,return_train_score=True,scoring='r2',n_jobs=-1)

result_2.fit(x,y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...Regressor())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__n_neighbors': [5, 7, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` paramete

In [29]:
print(result_2.best_score_)
print(result_2.best_params_)

0.9005232086831996
{'model__n_neighbors': 7}


In [30]:
tuned_pipeline = Pipeline(steps=[('preprocessing',preprocessing),
                               ('model',KNeighborsRegressor(n_neighbors=7))])
tuned_pipeline.fit(x,y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['country','capacity in MW','primary_fuel','geolocation_source']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('one_hot', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of 

In [31]:
joblib.dump(tuned_pipeline,'knn.pkl')

['knn.pkl']

In [102]:
%%writefile power_plants.py

import pandas as pd
import streamlit as st
import plotly.express as px
import joblib
import category_encoders 
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from category_encoders import BinaryEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# setting up streamlit dashboard and loading the data and the model 

st.set_page_config(layout='wide',page_title='Power Plants')
st.title('Global Power Plants')

df=pd.read_csv('global_power_clean.csv')

knn = joblib.load('knn.pkl')



# Dividing dashboard into multiple sections


# overview page

page=st.sidebar.radio('select page',['Overview','Analysis','Prediction'])

if page == 'Overview':
    st.image('https://images.unsplash.com/photo-1578776349090-de61da00ff1a?w=600&auto=format&fit=crop&q=60&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxzZWFyY2h8Mnx8cG93ZXIlMjBwbGFudHxlbnwwfHwwfHx8MA%3D%3D')
    st.dataframe(df.head(10))

    # explaining features meaning
    cols = {'country' :'the country where the power station is located',
                'capacity in MW':'The maximum electrical power the facility can produce in megawatts' ,
                'primary_fuel' :'the primary fuel/technology used (wind, hydro, solar)',
                'geolocation_source':'The source or method used to obtain the facility’s geographic coordinates—such as an official registry, satellite data, or manual geocoding. It indicates how the location was identified, not the location itself',
                'estimated_generation':'The predicted amount of electricity the facility generates in gigawatts'}
        
    for col, meaning in cols.items():
        with st.sidebar.expander(col):
            st.write(meaning)

# analysis and visualization page

elif page == 'Analysis' :
    tab1,tab2,tab3,tab4,tab5,tab6 = st.tabs(['KPIs','Distributions','Energy Resources','fuel vs. capacity & generation','Capacity vs. Generation',
                                        'Top 10'])
    
    with tab1 :

        total_countries = len(df['country'].unique())
        total_geo_sources = len(df['geolocation_source'].unique())
        avg_capacity = round(df['capacity in MW'].mean(),2)
        avg_generation = round(df['estimated_generation'].mean(),2)

        col1,col2 = st.columns(2,gap = 'large')
        col3,col4 = st.columns(2,gap = 'large') 
        col1.metric('Total Countries',total_countries)
        col2.metric('Total Geolocation Sources',total_geo_sources)
        col3.metric('Average Plant Capacity MW',avg_capacity)
        col4.metric('Average Estimated Generation GW',avg_generation)

    with tab2:
        
        col5,col6= st.columns(2,gap = 'large')
       

        with col5 :
            fig1 =px.histogram(df,x='capacity in MW',marginal = 'rug',nbins=5,
                        title="Distribution of Plant Capacity",color_discrete_sequence=['cornflowerblue'])
            st.plotly_chart(fig1,use_container_width=True)

        with col6 :
            fig2 =px.histogram(df,x='estimated_generation',marginal = 'rug',nbins=5,
                                    title='Distribution Eestimated Generation',color_discrete_sequence=['cornflowerblue'])
            st.plotly_chart(fig2,use_container_width=True)


    with tab3 :
        fig3 = px.histogram(df,x='primary_fuel',
                        title='Distribution of Primary Fuel',color_discrete_sequence=['dodgerblue'])
        st.plotly_chart(fig3,use_container_width=True)


    with tab4 :
        

        bar_1= px.bar(df,x = 'primary_fuel',y='capacity in MW',
            color_discrete_sequence=['dodgerblue'],barmode='group')
        st.plotly_chart(bar_1,use_container_width=True)

        bar_2= px.bar(df,x = 'primary_fuel',y='estimated_generation',barmode='group')
        st.plotly_chart(bar_2,use_container_width=True)

        st.markdown('''Hydro power plants are the top in both power capacity and generation, more efforts and focus should be shifted
                    towards solar and wind energy as they are abundunt and available in most countries and regions
                    ''')

    with tab5 :
        
        scatter = px.scatter(df,y='capacity in MW',x='estimated_generation',trendline='ols',
                     color_discrete_sequence=['steelblue'])
        st.plotly_chart(scatter,use_container_width=True)

        st.markdown('Capacity of power plant correlates strongly with estimated electricity generation')

    with tab6:
        

        group_1 = df.groupby('country')['capacity in MW'].mean().reset_index().sort_values(by='capacity in MW',ascending=False).head(10)

        bar_3 = px.bar(group_1,y = 'capacity in MW',x ='country',title="TOP 10 Countries by  Average Plant Capacity",
               color_discrete_sequence=['skyblue'])

        st.plotly_chart(bar_3,use_container_width=True)
     

        group_2 = df.groupby('country')['estimated_generation'].mean().reset_index().sort_values(by='estimated_generation',ascending=False).head(10)

        bar_4 = px.bar(group_2,y = 'estimated_generation',x ='country',title="TOP 10 Countries by  Average Estimated Generation",
               color_discrete_sequence=['cornflowerblue'])

        st.plotly_chart(bar_4,use_container_width=True)


# prediction page

else:
    st.image("https://plus.unsplash.com/premium_photo-1661898205432-d648667b9c76?w=600&auto=format&fit=crop&q=60&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxzZWFyY2h8MXx8cG93ZXIlMjBwbGFudHxlbnwwfHwwfHx8MA%3D%3D")
    

  
    

    # Filters

    st.sidebar.header('Filters')

    country = st.sidebar.selectbox('country',df['country'].unique())
    filtered_geo = df[df['country']==country]['geolocation_source'].unique()
    geo_location = st.sidebar.selectbox('Geolocation',filtered_geo)

    primary_fuel = st.sidebar.radio('Energy Source',df['primary_fuel'].unique())

    capacity = st.sidebar.slider('Power Plant Capacity MW',min_value=float(df['capacity in MW'].min()),
                                                    max_value= float(df['capacity in MW'].max()))
    # Prediction

    if st.button("Predict Electricity Generation"):
        df_prediction = pd.DataFrame(data= [[country,capacity,primary_fuel,geo_location]],
                    columns=df.drop('estimated_generation',axis=1).columns)
        try:
            st.table(df_prediction)
            result = knn.predict(df_prediction)[0]
            prediction = knn.predict(df_prediction)
            st.success(f" Estimated Energy Generation: {prediction[0]:,.2f} gw")
        except Exception as e:
            st.error(f"Prediction Error: {e}")










   

Overwriting power_plants.py


In [99]:
! streamlit run power_plants.py

^C
